In [1]:
def warn (*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

import requests
import wget



In [2]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')

100% [..........................................................] 15660 / 15660file downloaded


In [3]:
loader = TextLoader(filename) 
documents = loader.load() 
text_splitter = CharacterTextSplitter(chunk_size =1000, chunk_overlap = 0)
texts = text_splitter.split_documents(documents)
print(len(texts))

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


In [4]:
from langchain_community.vectorstores import Chroma
embeddings = HuggingFaceEmbeddings() 
docsearch = Chroma.from_documents(texts, embeddings)
print("document")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

document


In [5]:
import os 
import getpass 
def set_if_undefined(var:str):
    if os.environ.get(var): 
        return 
    os.environ[var] = getpass.getpass(var) 
set_if_undefined("GROQ_API_KEY")    

GROQ_API_KEY ········


In [6]:
groq_llm = ChatGroq(
    model = "llama-3.3-70b-versatile", 
    max_tokens = 1024, 
    temperature = 0.5, 
    model_kwargs= {
        "top_p": 0.8
    }
)

In [7]:
#create retriever 
retriever = docsearch.as_retriever() 

In [11]:
#Simple RAG prompt
rag_template = """Answer the question based only on the following context:

{context}

Question: {question}

Answer:"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)
# Basic RAG chain
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])
    

In [12]:
# Create RAG chain
rag_chain = (
    {"context": retriever| format_docs ,"question": RunnablePassthrough()}
    | rag_prompt
    | groq_llm
    |StrOutputParser()
)
query = """ Can you summarize the document fo me?"""
answer = rag_chain.invoke(query)
print(f" the query is {query}")
print(f" The answer is {answer}")

 the query is  Can you summarize the document fo me?
 The answer is The document appears to be a collection of policies for an organization. It includes a Recruitment Policy, a Health and Safety Policy, and an Anti-discrimination and Harassment Policy, as well as a Code of Conduct. The Health and Safety Policy prioritizes the well-being of employees, customers, and the public, while the Anti-discrimination and Harassment Policy and Code of Conduct emphasize the importance of integrity, respect, and accountability in the workplace. The Code of Conduct also touches on safety, environmental responsibility, and ethical conduct. Overall, the document outlines the organization's commitment to maintaining a safe, respectful, and responsible work environment.


In [13]:
query = """ Can i eat in company vehicle """
answer = rag_chain.invoke(query) 
answer

'The provided context does not explicitly mention eating in a company vehicle. It only mentions that smoking is not permitted in company vehicles to maintain their condition and cleanliness. There is no information regarding eating in company vehicles.'

In [15]:
prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definately do not try to make up an answer.

{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(prompt_template)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | groq_llm
    | StrOutputParser()
)

In [16]:
answer = rag_chain.invoke(query)
answer

"I don't know. The provided policies discuss smoking, health and safety, anti-discrimination and harassment, drug and alcohol use, and mobile phone usage, but they do not mention eating in company vehicles."

In [17]:
query = "What I cannot do in it?"
answer = rag_chain.invoke(query)
answer

'Based on the provided document, here are some things you cannot do:\n\n1. Share your login credentials or passwords.\n2. Use internet and email for harassment, discrimination, or to distribute offensive or inappropriate content.\n3. Transmit confidential information without encryption.\n4. Discuss company matters on public forums or social media without discretion.\n5. Use mobile devices for personal tasks that disrupt work obligations.\n6. Download apps or click on links from unfamiliar sources without caution.\n7. Transmit sensitive company information via unsecured messaging apps or emails.\n8. Use company-issued phones for personal charges without reimbursing the company.\n\nThese are some of the things you cannot do according to the Internet and Email Policy and the Mobile Phone Policy.'